In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import time
from pathlib import Path
from typing import Dict, Tuple, Optional, Union, Any, List
import warnings
import scanpy as sc
warnings.filterwarnings('ignore')

# Import additional libraries for neural network training
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.metrics import precision_recall_curve, roc_curve, roc_auc_score, average_precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Dropout, BatchNormalization, Input, Add, Activation, 
    MultiHeadAttention, LayerNormalization, Reshape, Flatten,
    GlobalAveragePooling1D, Embedding
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, LearningRateScheduler
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras import backend as K
from sklearn.ensemble import RandomForestClassifier

In [2]:
ref_data = pd.read_csv('./smart_aligned_v2_dataset_reference_SL.csv')
ref_data

,tcr,peptide,label,tcr_source_dataset,tcr_source_index,peptide_source_dataset,peptide_source_index,donor,binding_tcr
0,CASSLYEQYF,GILGFVFTL,1,10X,0,10X,0,donor1,Y
1,CAWTGTGKIGWDSPLHF,KLGGALQAK,1,10X,2,10X,2,donor1,Y
2,CASSWGGGSHYGYTF,IVTDFSVIK,1,10X,3,10X,3,donor1,Y
3,CASSLYSATGELFF,AVFDRKSDAK,1,10X,5,10X,5,donor1,Y
4,CASSLYSATGELFF,AVFDRKSDAK,1,10X,6,10X,6,donor1,Y
...,...,...,...,...,...,...,...,...,...
287359,CASAPGPGYEQYF,FLYALALLL,0,10X,3326,10X,134,NaN,N
287360,CASSEQGAGSNQPQHF,RAKFKQLL,0,10X,108479,10X,44,NaN,Y
287361,CASRTGLASTDTQYF,RLRAEAQVK,0,10X,138071,10X,15,NaN,Y
287362,CASSVAVGTGSGANVLTF,FLYALALLL,0,10X,56246,10X,134,NaN,N


In [3]:
get_10_X_dat = sc.read_h5ad('../data/merge_gex_all_donors_all_peptides_meta_for_Leah_dat.h5ad')

In [4]:
ref_gex = get_10_X_dat[ref_data['tcr_source_index'].values,:]
gex = ref_gex.X.toarray()

In [5]:
obs = get_10_X_dat.obs.copy()
obs = obs.reset_index().reset_index().rename(
    columns={'index': 'tcr_source_index'}  # this 'tcr_source_index' should match ref_data
)

obs_lookup = obs[['tcr_source_index', 'donor']]

ref_merged = ref_data.merge(
    obs_lookup,
    on='tcr_source_index',
    how='left',
    suffixes=('', '_from_obs')
)

ref_merged['donor'] = ref_merged['donor'].fillna(ref_merged['donor_from_obs'])
ref_merged = ref_merged.drop(columns=['donor_from_obs'])
ref_merged

,tcr,peptide,label,tcr_source_dataset,tcr_source_index,peptide_source_dataset,peptide_source_index,donor,binding_tcr
0,CASSLYEQYF,GILGFVFTL,1,10X,0,10X,0,donor1,Y
1,CAWTGTGKIGWDSPLHF,KLGGALQAK,1,10X,2,10X,2,donor1,Y
2,CASSWGGGSHYGYTF,IVTDFSVIK,1,10X,3,10X,3,donor1,Y
3,CASSLYSATGELFF,AVFDRKSDAK,1,10X,5,10X,5,donor1,Y
4,CASSLYSATGELFF,AVFDRKSDAK,1,10X,6,10X,6,donor1,Y
...,...,...,...,...,...,...,...,...,...
287359,CASAPGPGYEQYF,FLYALALLL,0,10X,3326,10X,134,donor1,N
287360,CASSEQGAGSNQPQHF,RAKFKQLL,0,10X,108479,10X,44,donor3,Y
287361,CASRTGLASTDTQYF,RLRAEAQVK,0,10X,138071,10X,15,donor4,Y
287362,CASSVAVGTGSGANVLTF,FLYALALLL,0,10X,56246,10X,134,donor2,N


In [6]:
batch_gex = pd.read_csv('10X_data_pca_harmony_batch_correction_by_donor_embeddings.csv', index_col=0)
batch_gex = batch_gex.loc[ref_merged['tcr_source_index'].values]
batch_gex

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,1.087143,0.467443,2.992532,-0.600697,-3.994596,-3.333098,1.494854,-1.146100,1.571670,-3.118363,...,-0.210213,1.269835,0.131614,1.555858,-0.562283,0.154132,1.219355,-0.089831,-0.300811,0.098589
2,-4.378371,1.180819,-4.223704,-5.470704,0.444927,-0.000247,1.059516,0.668836,0.961870,1.295324,...,1.722274,0.045716,0.730128,1.226174,1.399147,-0.674245,0.482494,0.890334,-1.046426,-0.332977
3,-0.850734,5.800984,2.319241,0.621932,0.391961,1.471914,-1.820410,-1.887836,0.645741,-0.318717,...,-0.187072,0.493142,-0.507019,-0.700821,-0.482881,1.702497,-2.204038,0.750383,2.090266,-0.414377
5,-0.790633,-0.208497,-1.233520,2.680560,1.708483,0.613669,-1.287696,0.153485,-0.021906,-0.508320,...,-0.138782,-0.516450,1.391405,0.140142,-0.495096,0.871033,-0.136164,0.289218,-0.143441,-0.316246
6,-0.529419,-0.319814,-2.077449,2.748021,1.083930,0.285122,-1.394146,-1.113498,1.634049,-2.110551,...,-0.390809,1.235467,-0.210995,0.477084,0.740880,1.041670,0.724317,-1.486853,0.464953,-1.606659
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3326,6.443489,-4.038393,1.308122,-0.403288,-0.090232,-1.258167,-0.057296,-0.714812,1.099047,-0.373782,...,0.566388,0.148645,-0.744329,0.656591,-0.616194,0.253673,-0.607318,0.485932,-0.095158,-0.634528
108479,0.596411,4.602098,3.885762,0.010277,-0.218515,2.740965,-0.837905,0.567612,0.722672,-2.788494,...,0.579572,-0.395867,1.981411,0.793389,0.018147,1.137729,1.211720,0.993538,-0.359124,1.103844
138071,-2.703015,10.533441,2.832771,-4.075355,2.132309,4.101437,-0.075762,-0.477110,0.912201,1.863241,...,1.218484,0.683383,1.670621,1.108839,2.624515,-1.777180,-0.853959,-0.634077,-0.251551,1.076117
56246,-1.658694,-0.560859,0.262945,-0.219867,-4.527000,0.595254,2.783486,0.530106,0.561798,0.377792,...,0.766667,0.233305,0.220572,-1.416050,-0.908362,0.383893,-1.113717,0.615160,-0.365468,-1.293682


# Create indices for Random Split

In [7]:
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temp (val+test)
train_data, temp_data = train_test_split(
    ref_data, 
    test_size=0.3, 
    stratify=ref_data['label'], 
    random_state=42
)

# Second split: split temp into 50-50 for val and test (15% each of original)
val_data, test_data = train_test_split(
    temp_data, 
    test_size=0.5, 
    stratify=temp_data['label'], 
    random_state=42
)

print(f"Train: {len(train_data)} ({len(train_data)/len(ref_data)*100:.1f}%)")
print(f"Val: {len(val_data)} ({len(val_data)/len(ref_data)*100:.1f}%)")
print(f"Test: {len(test_data)} ({len(test_data)/len(ref_data)*100:.1f}%)")

Train: 201154 (70.0%)
Val: 43105 (15.0%)
Test: 43105 (15.0%)


In [8]:
np.save('./smart_aligned_v2_train_indices_random_split_1.npy', train_data.index)
np.save('./smart_aligned_v2_val_indices_random_split_1.npy', val_data.index)
np.save('./smart_aligned_v2_test_indices_random_split_1.npy', test_data.index)

In [9]:
train_indices = np.load('./smart_aligned_v2_train_indices_random_split_1.npy')
val_indices = np.load('./smart_aligned_v2_val_indices_random_split_1.npy')
test_indices = np.load('./smart_aligned_v2_test_indices_random_split_1.npy')

# Create indices for TCR Split

In [10]:
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

def create_stratified_group_split(data, test_size=0.2, val_size=0.2, random_state=42):
    """
    Creates train, validation, and test splits that are stratified by label 
    and grouped by TCR to prevent overlap.
    
    Args:
        data (pd.DataFrame): The full dataset to split.
        test_size (float): The proportion of the dataset to allocate to the test set.
        val_size (float): The proportion of the train+val set to allocate to the validation set.
                           (e.g., 0.25 of the remaining 80% is 20% of the total).
        random_state (int): The random seed for reproducibility.
    """
    
    # n_splits is calculated to achieve the desired test_size
    # For test_size=0.2, n_splits=5 (1/5). For 0.1, n_splits=10 (1/10).
    n_splits = int(1 / test_size)
    
    # Initialize the splitter
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    # Get the labels (y) and groups for the split
    labels = data['label']
    groups = data['tcr']
    
    # Generate splits and take the first one
    train_val_idx, test_idx = next(sgkf.split(data, labels, groups))
    
    data_train_val = data.iloc[train_val_idx]
    data_test = data.iloc[test_idx]
    
    # Now, create the train and validation sets from the train_val data
    # We stratify this split as well to maintain the class balance
    data_train, data_val = train_test_split(
        data_train_val,
        test_size=val_size,
        random_state=random_state,
        shuffle=True,
        stratify=data_train_val['label']
    )
    
    # --- Report and verify the splits ---
    print("Split Summary:")
    for d, name in [(data_train, 'Training'), (data_val, 'Validation'), (data_test, 'Test')]:
        n_samples = len(d)
        pos_percent = d['label'].mean() * 100
        n_tcrs = d['tcr'].nunique()
        print(f"  - {name} Set: {n_samples:,} samples ({pos_percent:.1f}% positive) from {n_tcrs:,} unique TCRs")
        
    # Verify no TCR overlap between train/val and test
    tcrs_train_val = set(data_train_val['tcr'].unique())
    tcrs_test = set(data_test['tcr'].unique())
    overlap = len(tcrs_train_val.intersection(tcrs_test))
    print(f"\nOverlap of TCRs between Train and Val: {len(data_train['tcr'].unique()) + len(data_val['tcr'].unique()) - len(tcrs_train_val)} (allowed)")
    print(f"\nOverlap of TCRs between (Train + Val) and Test: {overlap} (Should be 0)")
    
    print("\nTotal Samples Check:")
    total_samples = len(data_train) + len(data_val) + len(data_test)
    print(f"  - Total samples in splits: {total_samples:,}")
    
    return data_train, data_val, data_test

train_df, val_df, test_df = create_stratified_group_split(
    ref_data, 
    test_size=0.2, 
    val_size=0.2 
)

# Get the indices for slicing embeddings/labels
train_indices = train_df.index.values
val_indices = val_df.index.values
test_indices = test_df.index.values

# Shuffle indices to avoid order bias
np.random.seed(42)
np.random.shuffle(train_indices)
np.random.shuffle(val_indices)
np.random.shuffle(test_indices)

np.save('./smart_aligned_v2_train_indices_tcr_split_1.npy', train_indices)
np.save('./smart_aligned_v2_val_indices_tcr_split_1.npy', val_indices)
np.save('./smart_aligned_v2_test_indices_tcr_split_1.npy', test_indices)

# train_labels = ref_data.loc[train_indices, 'label'].values
# val_labels = data_train.loc[val_indices, 'label'].values
# test_labels = data_train.loc[test_indices, 'label'].values

Split Summary:
  - Training Set: 178,718 samples (25.0% positive) from 36,318 unique TCRs
  - Validation Set: 44,680 samples (25.0% positive) from 17,025 unique TCRs
  - Test Set: 63,966 samples (25.0% positive) from 9,702 unique TCRs

Overlap of TCRs between Train and Val: 14735 (allowed)

Overlap of TCRs between (Train + Val) and Test: 0 (Should be 0)

Total Samples Check:
  - Total samples in splits: 287,364


In [11]:
# ref_data = pd.read_csv('./smart_aligned_v2_dataset_reference.csv')
train_indices = np.load('./smart_aligned_v2_train_indices_tcr_split_1.npy')
val_indices = np.load('./smart_aligned_v2_val_indices_tcr_split_1.npy')
test_indices = np.load('./smart_aligned_v2_test_indices_tcr_split_1.npy')
train_raw_data = ref_data.iloc[train_indices]
val_raw_data = ref_data.iloc[val_indices]
test_raw_data = ref_data.iloc[test_indices]

In [12]:
train_raw_data

,tcr,peptide,label,tcr_source_dataset,tcr_source_index,peptide_source_dataset,peptide_source_index,donor,binding_tcr
161061,CASSQEQGLAYIQYF,RAKFKQLL,0,10X,120118,10X,44,NaN,Y
189440,CASSIGLYGYTF,KLGGALQAK,0,10X,85739,10X,2,NaN,Y
101677,CASSLTRRYTF,AVFDRKSDAK,0,10X,43848,10X,5,NaN,N
272605,CASSLGGGGYNEQFF,IVTDFSVIK,0,10X,94476,10X,3,NaN,N
218856,CSVGTGDWGEQYF,RLRAEAQVK,0,10X,45294,10X,15,NaN,Y
...,...,...,...,...,...,...,...,...,...
238843,CSARDVSYNEQFF,KLGGALQAK,0,10X,134710,10X,2,NaN,N
231571,CASSHTSADEQFF,RAKFKQLL,0,10X,139068,10X,44,NaN,N
189028,CSARDLLAGDTDTQYF,GLCTLVAML,0,10X,87640,10X,1116,NaN,N
136683,CSARDPRESSYEQYF,ELAGIGILTV,0,10X,34546,10X,258,NaN,N


# Create indices for Donor Split

In [13]:
ref_merged.donor.value_counts()

donor
donor2    110678
donor1     78178
donor3     76297
donor4     22211
Name: count, dtype: int64

In [14]:
donors = ref_merged['donor'].unique()
assert len(donors) == 4, "Expected exactly 4 unique donors for LODO splits."

output_dir = './'

for i, test_donor in enumerate(donors):
    print(f"\n=== Split {i+1}: Test on donor {test_donor} ===")
    
    # Split by donor
    test_mask = ref_merged['donor'] == test_donor
    train_val_mask = ~test_mask
    
    test_data = ref_merged[test_mask].copy()
    train_val_data = ref_merged[train_val_mask].copy()
    
    print(f"Test donor {test_donor}: {len(test_data)} samples ({len(test_data)/len(ref_merged)*100:.1f}%)")
    print(f"Train+Val (other 3 donors): {len(train_val_data)} samples ({len(train_val_data)/len(ref_merged)*100:.1f}%)")
    
    train_data, val_data = train_test_split(
        train_val_data, 
        test_size=0.25,
        stratify=train_val_data['label'], 
        random_state=42
    )
    
    print(f"  Train: {len(train_data)} ({len(train_data)/len(ref_merged)*100:.1f}%)")
    print(f"  Val:   {len(val_data)} ({len(val_data)/len(ref_merged)*100:.1f}%)")
    
    # Save indices (relative to original ref_merged)
    np.save(f'{output_dir}smart_aligned_v2_train_indices_donor_LOO_{test_donor}.npy', train_data.index.values)
    np.save(f'{output_dir}smart_aligned_v2_val_indices_donor_LOO_{test_donor}.npy', val_data.index.values)
    np.save(f'{output_dir}smart_aligned_v2_test_indices_donor_LOO_{test_donor}.npy', test_data.index.values)

print("\n✅ All 4 LODO splits saved!")


=== Split 1: Test on donor donor1 ===
Test donor donor1: 78178 samples (27.2%)
Train+Val (other 3 donors): 209186 samples (72.8%)
  Train: 156889 (54.6%)
  Val:   52297 (18.2%)

=== Split 2: Test on donor donor2 ===
Test donor donor2: 110678 samples (38.5%)
Train+Val (other 3 donors): 176686 samples (61.5%)
  Train: 132514 (46.1%)
  Val:   44172 (15.4%)

=== Split 3: Test on donor donor3 ===
Test donor donor3: 76297 samples (26.6%)
Train+Val (other 3 donors): 211067 samples (73.4%)
  Train: 158300 (55.1%)
  Val:   52767 (18.4%)

=== Split 4: Test on donor donor4 ===
Test donor donor4: 22211 samples (7.7%)
Train+Val (other 3 donors): 265153 samples (92.3%)
  Train: 198864 (69.2%)
  Val:   66289 (23.1%)

✅ All 4 LODO splits saved!
